# Tajweed Analysis Model Training on Kaggle TPU

**Architecture**: CNN-LSTM for 4-class Tajweed classification

**Classes** (from QDAT dataset):
- 0: Correct (no error)
- 1: Separate tide (المد المنفصل)
- 2: The tight noon (النون المشددة  )
- 3: Concealment (الإخفاء)

**Target**: 90-93% accuracy in 10-15 min on TPU v5e-8

## 1. Setup & TPU Initialization

In [ ]:
# Install only essential packages (avoid transformers to prevent tokenizers build error)
%pip install -q librosa soundfile pandas scikit-learn


In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import librosa
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils import class_weight
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# TPU Strategy Setup
try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    print("Device:", tpu.master())
    strategy = tf.distribute.TPUStrategy(tpu)
    print(f"Number of replicas: {strategy.num_replicas_in_sync}")
except ValueError:
    print("TPU not found, using default strategy (GPU/CPU)")
    strategy = tf.distribute.get_strategy()

print(f"Using strategy: {type(strategy).__name__}")
print(f"Num replicas: {strategy.num_replicas_in_sync}")

## 2. Configuration

In [ ]:
# Training Configuration
CONFIG = {
    # Model
    'num_classes': 4,  # Correct, Separate tide, Tight noon, Concealment
    
    # Audio
    'sample_rate': 16000,
    'max_duration': 10,  # seconds
    'n_mfcc': 40,  # MFCC coefficients
    'n_fft': 2048,
    'hop_length': 512,
    
    # Training
    'batch_size': 32,
    'epochs': 12,
    'learning_rate': 0.001,
    'label_smoothing': 0.1,
    
    # Augmentation
    'augment_prob': 0.5,
    'noise_level': 0.005,
    'speed_range': (0.9, 1.1),
    
    # Paths (adjust based on your Kaggle dataset)
    'csv_path': '/kaggle/input/qdat/qdat_dataset.csv',
    'audio_dir': '/kaggle/input/qdat/audio',
    'output_dir': '/kaggle/working',
    'model_save_path': '/kaggle/working/tajweed_model.keras',
    'tflite_path': '/kaggle/working/tajweed_model.tflite',
}

# Enable mixed precision for TPU
tf.keras.mixed_precision.set_global_policy('mixed_bfloat16')

# Enable XLA
tf.config.optimizer.set_jit(True)

print(json.dumps(CONFIG, indent=2))

## 3. Dataset Loading & Preprocessing

In [ ]:
# Load QDAT CSV
try:
    df = pd.read_csv(CONFIG['csv_path'])
    print(f"Dataset loaded: {len(df)} samples")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head())
except FileNotFoundError:
    print("QDAT dataset not found. Creating dummy dataset...")
    # Create dummy data for testing
    df = pd.DataFrame({
        'audio_file': [f'audio_{i:04d}.wav' for i in range(1000)],
        'class': np.random.choice(['Correct', 'Separate tide', 'The tight noon', 'Concealment'], 1000),
        'target': np.random.randint(0, 2, 1000)
    })
    print("Using dummy dataset for demonstration")

In [ ]:
# Map classes to integer labels
# Assuming the CSV has a column indicating the Tajweed class
# Adjust column names based on your actual CSV structure

class_mapping = {
    'Correct': 0,
    'Separate tide': 1,
    'The tight noon': 2,
    'Concealment': 3
}

class_names = ['Correct', 'Separate tide', 'The tight noon', 'Concealment']

# Create labels based on CSV structure
# If you have a 'class' column with class names:
if 'class' in df.columns:
    df['label'] = df['class'].map(class_mapping)
# Or if you have separate columns, adjust accordingly:
# df['label'] = ...

# Remove any rows with missing labels
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)  # Ensure integer dtype

print(f"\nClass distribution:")
print(df['label'].value_counts().sort_index())
print(f"\nClass percentages:")
print((df['label'].value_counts(normalize=True).sort_index() * 100).round(2))

In [ ]:
# Train/Val/Test split
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df['label'], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nTrain distribution: {train_df['label'].value_counts().sort_index().tolist()}")
print(f"Val distribution: {val_df['label'].value_counts().sort_index().tolist()}")
print(f"Test distribution: {test_df['label'].value_counts().sort_index().tolist()}")

# Calculate class weights for handling imbalance
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_df['label']),
    y=train_df['label']
)
class_weights_dict = dict(enumerate(class_weights))
print(f"\nClass weights: {class_weights_dict}")

## 4. Feature Extraction (MFCC)

In [ ]:
def extract_mfcc(audio_path, sr=16000, n_mfcc=40, max_len=None):
    """
    Extract MFCC features from audio file.
    """
    try:
        # Load audio
        y, sr = librosa.load(audio_path, sr=sr, duration=CONFIG['max_duration'])
        
        # Extract MFCCs
        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=n_mfcc,
            n_fft=CONFIG['n_fft'],
            hop_length=CONFIG['hop_length']
        )
        
        # Pad or truncate to fixed length
        if max_len is not None:
            if mfcc.shape[1] < max_len:
                pad_width = max_len - mfcc.shape[1]
                mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='constant')
            else:
                mfcc = mfcc[:, :max_len]
        
        return mfcc.T  # Transpose to (time_steps, n_mfcc)
    
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

def augment_audio(y, sr=16000):
    """
    Apply random augmentations to audio.
    """
    if np.random.random() < CONFIG['augment_prob']:
        # Add noise
        if np.random.random() < 0.5:
            noise = np.random.randn(len(y))
            y = y + CONFIG['noise_level'] * noise
        
        # Time stretch
        if np.random.random() < 0.5:
            rate = np.random.uniform(*CONFIG['speed_range'])
            y = librosa.effects.time_stretch(y, rate=rate)
    
    return y

# Test feature extraction
print("Testing feature extraction...")
sample_mfcc = extract_mfcc(
    os.path.join(CONFIG['audio_dir'], train_df.iloc[0]['audio_file']) if 'audio_file' in train_df.columns else 'dummy.wav',
    max_len=311  # ~10 seconds at 16kHz
)
if sample_mfcc is not None:
    print(f"MFCC shape: {sample_mfcc.shape}")

In [ ]:
# Determine max sequence length from training data
print("Calculating optimal sequence length...")
seq_lengths = []
for i in range(min(100, len(train_df))):
    audio_file = train_df.iloc[i]['audio_file'] if 'audio_file' in train_df.columns else f'audio_{i:04d}.wav'
    audio_path = os.path.join(CONFIG['audio_dir'], audio_file)
    try:
        y, sr = librosa.load(audio_path, sr=CONFIG['sample_rate'], duration=CONFIG['max_duration'])
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=CONFIG['n_mfcc'])
        seq_lengths.append(mfcc.shape[1])
    except:
        continue

if seq_lengths:
    max_seq_len = int(np.percentile(seq_lengths, 95))  # Use 95th percentile
else:
    max_seq_len = 311  # Default for 10 seconds

print(f"Using max sequence length: {max_seq_len}")
CONFIG['max_seq_len'] = max_seq_len

## 5. Create TensorFlow Datasets

In [ ]:
def load_data(df, augment=False):
    """
    Load audio files and extract features.
    """
    X = []
    y = []
    
    for idx, row in df.iterrows():
        # Get audio file path
        audio_file = row['audio_file'] if 'audio_file' in row else f'audio_{idx:04d}.wav'
        audio_path = os.path.join(CONFIG['audio_dir'], audio_file)
        
        try:
            # Extract features
            mfcc = extract_mfcc(
                audio_path,
                sr=CONFIG['sample_rate'],
                n_mfcc=CONFIG['n_mfcc'],
                max_len=CONFIG['max_seq_len']
            )
            
            if mfcc is not None:
                X.append(mfcc)
                y.append(row['label'])
        except Exception as e:
            print(f"Error loading {audio_file}: {e}")
            continue
    
    return np.array(X), np.array(y, dtype=np.int32)

print("Loading training data...")
X_train, y_train = load_data(train_df, augment=True)
print(f"Train: X={X_train.shape}, y={y_train.shape}")

print("\nLoading validation data...")
X_val, y_val = load_data(val_df, augment=False)
print(f"Val: X={X_val.shape}, y={y_val.shape}")

print("\nLoading test data...")
X_test, y_test = load_data(test_df, augment=False)
print(f"Test: X={X_test.shape}, y={y_test.shape}")

# Reshape for Conv1D: (samples, time_steps, features)
print(f"\n Train shape: {X_train.shape}")
print(f"Val shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

## 6. Build CNN-LSTM Model

In [ ]:
def build_cnn_lstm_model(input_shape, num_classes):
    """
    Build CNN-LSTM model for audio classification.
    
    Architecture:
    - Conv1D layers for feature extraction
    - Bidirectional LSTM for temporal modeling
    - Dense layers for classification
    """
    with strategy.scope():
        model = tf.keras.Sequential([
            # Input
            tf.keras.layers.Input(shape=input_shape, name='input'),
            
            # CNN blocks
            tf.keras.layers.Conv1D(128, 5, padding='same', activation='relu', name='conv1'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Dropout(0.2),
            
            tf.keras.layers.Conv1D(256, 5, padding='same', activation='relu', name='conv2'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Dropout(0.2),
            
            tf.keras.layers.Conv1D(128, 3, padding='same', activation='relu', name='conv3'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.MaxPooling1D(2),
            tf.keras.layers.Dropout(0.2),
            
            # Bidirectional LSTM
            tf.keras.layers.Bidirectional(
                tf.keras.layers.LSTM(128, return_sequences=True, name='lstm1')
            ),
            tf.keras.layers.Dropout(0.3),
            
            tf.keras.layers.Bidirectional(
                tf.keras.layers.LSTM(64, name='lstm2')
            ),
            tf.keras.layers.Dropout(0.3),
            
            # Dense layers
            tf.keras.layers.Dense(128, activation='relu', name='dense1'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.4),
            
            tf.keras.layers.Dense(64, activation='relu', name='dense2'),
            tf.keras.layers.Dropout(0.3),
            
            # Output
            tf.keras.layers.Dense(
                num_classes,
                activation='softmax',
                dtype='float32',  # Ensure float32 for output
                name='output'
            )
        ], name='TajweedCNNLSTM')
        
        return model

# Build model
input_shape = (X_train.shape[1], X_train.shape[2])  # (time_steps, n_mfcc)
print(f"Input shape: {input_shape}")

model = build_cnn_lstm_model(input_shape, CONFIG['num_classes'])
model.summary()

In [ ]:
# Count parameters
trainable_params = sum([tf.size(var).numpy() for var in model.trainable_variables])
total_params = sum([tf.size(var).numpy() for var in model.variables])

print(f"\nTrainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")

## 7. Training Setup & Execution

In [ ]:
with strategy.scope():
    # Optimizer with learning rate schedule
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=CONFIG['learning_rate'],
        decay_steps=1000,
        decay_rate=0.9
    )
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)
    
    # Loss with label smoothing
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        label_smoothing=CONFIG['label_smoothing']
    )
    
    # Compile
    model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=[
            'accuracy',
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_accuracy')
        ]
    )

print("Model compiled!")

In [ ]:
# Callbacks
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        CONFIG['model_save_path'],
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

In [ ]:
import time

print("Starting training...")
start_time = time.time()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    class_weight=class_weights_dict,
    callbacks=callbacks,
    verbose=1
)

training_time = time.time() - start_time
print(f"\n✅ Training completed in {training_time/60:.2f} minutes")

## 8. Evaluation & Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Accuracy
axes[0].plot(history.history['accuracy'], label='Train Accuracy', marker='o')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', marker='s')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'], label='Train Loss', marker='o')
axes[1].plot(history.history['val_loss'], label='Val Loss', marker='s')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

# Best metrics
best_epoch = np.argmax(history.history['val_accuracy'])
print(f"\n📊 Best Performance:")
print(f"  Epoch: {best_epoch + 1}")
print(f"  Val Accuracy: {history.history['val_accuracy'][best_epoch]:.4f}")
print(f"  Val Loss: {history.history['val_loss'][best_epoch]:.4f}")

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_loss, test_acc, test_top2 = model.evaluate(X_test, y_test, verbose=1)

print(f"\n📊 Test Results:")
print(f"  Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"  Top-2 Accuracy: {test_top2:.4f}")
print(f"  Loss: {test_loss:.4f}")

In [ ]:
# Predictions & Confusion Matrix
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

# Classification report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    cbar_kws={'label': 'Count'}
)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Export for Production

In [ ]:
# Save Keras model
print("Saving Keras model...")
model.save(CONFIG['model_save_path'], save_format='keras')
model_size = os.path.getsize(CONFIG['model_save_path']) / (1024 * 1024)
print(f"✅ Saved to {CONFIG['model_save_path']} ({model_size:.2f} MB)")

In [ ]:
# Convert to TFLite
print("\nConverting to TFLite...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

with open(CONFIG['tflite_path'], 'wb') as f:
    f.write(tflite_model)

tflite_size = len(tflite_model) / (1024 * 1024)
print(f"✅ TFLite model saved: {CONFIG['tflite_path']} ({tflite_size:.2f} MB)")
print(f"   Size reduction: {((model_size - tflite_size) / model_size * 100):.1f}%")

In [ ]:
# Test TFLite inference
interpreter = tf.lite.Interpreter(model_path=CONFIG['tflite_path'])
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"\n⚡ TFLite Inference Test:")
print(f"   Input shape: {input_details[0]['shape']}")
print(f"   Output shape: {output_details[0]['shape']}")

# Measure latency
test_input = X_test[0:1].astype(np.float32)

# Warmup
for _ in range(10):
    interpreter.set_tensor(input_details[0]['index'], test_input)
    interpreter.invoke()

# Measure
import time
latencies = []
for _ in range(100):
    start = time.time()
    interpreter.set_tensor(input_details[0]['index'], test_input)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    latencies.append((time.time() - start) * 1000)

avg_latency = np.mean(latencies)
std_latency = np.std(latencies)

print(f"\n📊 Performance:")
print(f"   Average latency: {avg_latency:.2f} ± {std_latency:.2f} ms")
print(f"   Throughput: {1000/avg_latency:.1f} inferences/sec")
print(f"   {'✅ Real-time capable!' if avg_latency < 100 else '⚠️ May be slow for real-time'}")

## 10. Save Metadata

In [ ]:
# Save model metadata
metadata = {
    'model_name': 'Tajweed CNN-LSTM Classifier',
    'version': '1.0',
    'architecture': 'CNN-LSTM',
    'num_classes': CONFIG['num_classes'],
    'class_names': class_names,
    'sample_rate': CONFIG['sample_rate'],
    'n_mfcc': CONFIG['n_mfcc'],
    'max_duration': CONFIG['max_duration'],
    'max_seq_len': CONFIG['max_seq_len'],
    'training': {
        'epochs': CONFIG['epochs'],
        'batch_size': CONFIG['batch_size'],
        'learning_rate': CONFIG['learning_rate'],
        'train_samples': len(train_df),
        'val_samples': len(val_df),
        'test_samples': len(test_df),
        'training_time_minutes': round(training_time / 60, 2),
        'class_weights': class_weights_dict
    },
    'performance': {
        'test_accuracy': float(test_acc),
        'test_top2_accuracy': float(test_top2),
        'test_loss': float(test_loss),
        'best_val_accuracy': float(history.history['val_accuracy'][best_epoch]),
        'tflite_latency_ms': round(avg_latency, 2),
        'tflite_size_mb': round(tflite_size, 2),
        'keras_size_mb': round(model_size, 2)
    },
    'created_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

metadata_path = os.path.join(CONFIG['output_dir'], 'model_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print("\n📋 Model Metadata:")
print(json.dumps(metadata, indent=2))
print(f"\n✅ Saved to {metadata_path}")

## 11. Summary

In [ ]:
print("\n" + "="*70)
print("  TRAINING COMPLETE - TAJWEED CNN-LSTM MODEL")
print("="*70)
print(f"\n📊 Final Results:")
print(f"   Test Accuracy: {test_acc*100:.2f}%")
print(f"   Training Time: {training_time/60:.2f} minutes")
print(f"   TFLite Latency: {avg_latency:.2f} ms")
print(f"   Model Size: Keras {model_size:.2f} MB → TFLite {tflite_size:.2f} MB")
print(f"\n📦 Saved Artifacts:")
print(f"   - Keras Model: {CONFIG['model_save_path']}")
print(f"   - TFLite Model: {CONFIG['tflite_path']}")
print(f"   - Metadata: {metadata_path}")
print(f"   - Training Plot: training_history.png")
print(f"   - Confusion Matrix: confusion_matrix.png")
print(f"\n🚀 Next Steps:")
print(f"   1. Download models from /kaggle/working/")
print(f"   2. Copy to backend-flask/models/")
print(f"   3. Update MODEL_PATH in .env")
print(f"   4. Test with real audio samples")
print(f"   5. Deploy to production")
print("\n" + "="*70)